# Sprint 3 Live Demo - Google Colab Setup

This notebook sets up the LLM demo in Google Colab.

**What this does:**
- Installs dependencies (llama-cpp-python, streamlit, boto3)
- Downloads Phi-3 Mini model (2.4GB)
- Sets up Streamlit dashboard
- Runs interactive demo

**Time:** ~5-8 minutes (model download takes 3-4 min)


In [ ]:
# Install dependencies
!pip install -q llama-cpp-python==0.2.20 streamlit==1.29.0 boto3==1.34.10 pandas==2.1.4 plotly==5.18.0 requests==2.31.0 tqdm==4.66.1


In [ ]:
# Download model
import os
import requests
from tqdm import tqdm

MODEL_URL = "https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf"
MODEL_DIR = "./models"
MODEL_NAME = "Phi-3-mini-4k-instruct-q4.gguf"
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_NAME)

os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    print(f"Downloading {MODEL_NAME} (2.4GB)...")
    print("This will take 3-5 minutes...")
    
    response = requests.get(MODEL_URL, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(MODEL_PATH, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    
    print(f"\nModel downloaded: {MODEL_PATH}")
else:
    size_mb = os.path.getsize(MODEL_PATH) / 1024 / 1024
    print(f"Model already exists: {MODEL_PATH} ({size_mb:.1f} MB)")

# Set environment variable
os.environ['MODEL_PATH'] = MODEL_PATH
print(f"\nModel path set to: {MODEL_PATH}")


In [ ]:
# Test LLM (requires utils/llm_runner.py to be uploaded)
# For Colab, you need to upload the live-demo directory files first

import sys
import os

# Add current directory to path
sys.path.append('.')

try:
    from utils.llm_runner import LLMRunner
    
    print("Loading LLM model...")
    llm = LLMRunner(model_path=MODEL_PATH)
    
    if llm.is_ready():
        print("PASS: LLM loaded successfully!")
        
        # Test with sample vitals
        test_vitals = {
            'patient_id': 'TEST-001',
            'heart_rate': 110,
            'bp_systolic': 145,
            'bp_diastolic': 92,
            'oxygen_saturation': 94.0,
            'respiratory_rate': 22,
            'temperature': 37.8
        }
        
        print("\nTesting LLM inference...")
        result = llm.analyze_vitals(test_vitals)
        
        print(f"\nPASS: Analysis complete!")
        print(f"Urgency: {result['urgency']}")
        print(f"Primary Concern: {result['primary_concern']}")
        print(f"Reasoning: {result['reasoning']}")
        print(f"Inference Time: {result['inference_time']:.3f}s")
        print(f"Method: {result['analysis_method']}")
    else:
        print(f"FAIL: LLM failed to load: {llm.load_error}")
        print("\nMake sure you've uploaded utils/llm_runner.py to Colab")
except ImportError as e:
    print(f"FAIL: Import error: {e}")
    print("\nTo use LLM in Colab:")
    print("1. Upload the 'live-demo' folder to Colab")
    print("2. Or clone the repo: !git clone https://github.com/FaustoRosado/AIER-alerts.git")
    print("3. Then run: cd AIER-alerts/tech-execution/sprint3/live-demo")
